# Lesson 05 — Vector Search

We have caption embeddings in S3 Vectors from Lesson 04. Now we search them **by text**.

The idea:
1. Encode a text query with CLIP → get a 512-d vector
2. Ask S3 Vectors for the nearest caption vectors
3. Return the top-k matching images and their captions

No Batch job this lesson — CLIP's text encoder runs locally on CPU and S3
Vectors performs the nearest-neighbour search.

> **No local GPU?** That's fine. CLIP's text encoder is small and runs on CPU in <1 second.

## Step 1 — Load environment and connect to S3 Vectors

In [ ]:
import io, os
import boto3
from dotenv import load_dotenv

load_dotenv(dotenv_path="../../.env")
S3_BUCKET = os.environ["S3_BUCKET"]
S3_VECTOR_BUCKET = os.environ["S3_VECTOR_BUCKET"]
S3_VECTOR_INDEX = os.environ["S3_VECTOR_INDEX"]

s3 = boto3.client("s3")
s3vectors = boto3.client("s3vectors")

print(f"Image bucket : {S3_BUCKET}")
print(f"Vector index : {S3_VECTOR_BUCKET}/{S3_VECTOR_INDEX}")

## Step 2 — Load the CLIP text encoder

In [ ]:
import torch
from transformers import CLIPModel, CLIPProcessor

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
model.eval()
print("CLIP loaded.")

## Step 3 — Search helper

This function encodes a text query, then asks S3 Vectors for the top matching captions/images.

In [ ]:
def search(query: str, top_k: int = 3) -> list[dict]:
    """Return the top_k S3 Vectors matches for a plain-English query."""
    # Encode the query text
    with torch.no_grad():
        inputs   = processor(text=[query], return_tensors="pt", padding=True).to(device)
        text_vec = model.get_text_features(**inputs)                        # shape: (1, 512)
        text_vec = text_vec / text_vec.norm(dim=-1, keepdim=True)           # normalise

    response = s3vectors.query_vectors(
        vectorBucketName=S3_VECTOR_BUCKET,
        indexName=S3_VECTOR_INDEX,
        topK=top_k,
        queryVector={"float32": text_vec.cpu().numpy().flatten().astype("float32").tolist()},
        returnMetadata=True,
        returnDistance=True,
    )
    return response["vectors"]

print("Search function ready.")

## Step 4 — Search! (edit the query and re-run)

Try: `"outdoor scene"`, `"close-up face"`, `"text on screen"`, `"dark scene"`, `"a busy street"`

In [ ]:
QUERY = "outdoor scene with trees"   # ← change me!

matches = search(QUERY, top_k=3)

print(f"Query: '{QUERY}'")
for rank, match in enumerate(matches, start=1):
    metadata = match["metadata"]
    print(f"  #{rank}  {metadata['image_key']}  caption={metadata['caption']!r}  distance={match['distance']:.3f}")

## Step 5 — Show the top-3 matching images

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

for ax, match in zip(axes, matches):
    metadata = match["metadata"]
    key = metadata["image_key"]
    obj = s3.get_object(Bucket=S3_BUCKET, Key=key)
    img = Image.open(io.BytesIO(obj["Body"].read()))
    ax.imshow(img)
    ax.set_title(f"{metadata['caption']}\nDistance: {match['distance']:.3f}", fontsize=9, wrap=True)
    ax.axis("off")

plt.suptitle(f"Top-3 results for: '{QUERY}'", fontsize=12)
plt.tight_layout()
plt.show()

## Key Takeaway

> We searched through all images using **plain English**, with no labels or
> keyword index. The GPU created the BLIP captions and CLIP embeddings, and
> S3 Vectors retrieved the nearest matching captions.

This pattern — caption → embed → store → search by cosine similarity — powers semantic search, RAG, and image search at every major tech company.

---

## Next lesson → [06 — Full Pipeline](../06-full-pipeline/notebook.ipynb)

We'll string lessons 03 and 04 together with Batch job dependencies — one command runs the full pipeline.